# Notebook to make some test about t2s of OpenAI models

## Libraries

In [19]:
import os
from dotenv import load_dotenv

from openai import AsyncOpenAI
from openai.helpers import LocalAudioPlayer
from openai import OpenAI

from IPython.display import Audio

import fitz  # PyMuPDF
import textwrap
from pydub import AudioSegment

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
openai = AsyncOpenAI(api_key=OPENAI_API_KEY)

In [7]:
client = OpenAI(api_key=OPENAI_API_KEY)
speech_file_path = "../Audios/speech.mp3"

## Test 1 (String to audio)

In [10]:
with client.audio.speech.with_streaming_response.create(
    model="gpt-4o-mini-tts",  # modelo económico y rápido
    voice="alloy",            # voz default (puedes cambiarla)
    input="Hola, esta es una prueba de Text to Speech con OpenAI."
) as response:
    response.stream_to_file(speech_file_path)

Audio(speech_file_path)

## Converting PDF to strings

In [12]:
# Abre el archivo PDF
doc = fitz.open("../PDFs/Dostoyevski, Fëdor - El jugador - Cap1.pdf")

# Itera por cada página
for num, page in enumerate(doc, start=1):
    text = page.get_text()
    print(f"--- Página {num} ---")
    print(text)
    print("\n")

doc.close()

--- Página 1 ---
CAPITULO I
 Por fin estaba de regreso, después de dos semanas de ausencia.
 Los nuestros llevaban ya tres días en Ruletenburg. Yo creía que meestarían aguardando como al
Mesías; pero me equivocaba. El general,que me recibió indiferente, me habló con altanería y me envió a
suhermana. Era evidente que, fuese como fuese, habían conseguido algún préstamo. Hasta me pareció
que el general rehuía mis miradas.
 María Philippovna, muy atareada, apenas si dijo unas palabras. Sinembargo, aceptó el dinero que le
traía, lo contó y escuchó mi relatohasta el fin. Estaban invitados a comer Mezontsov, un francés y
también un inglés. Desde luego, aquí, cuando se tiene dinero, se ofrece ungran banquete a los amigos.
Costumbre moscovita.
 Paulina Alexandrovna, al verme, me preguntó en seguida porqué había tardado tanto en volver, y sin
esperar mi respuesta se retiróinmediatamente. Naturalmente que aquello lo hizo adrede. Pero
eraindispensable, sin embargo, tener una explicación. Tengo el 

In [ ]:
doc = fitz.open("../PDFs/Dostoyevski, Fëdor - El jugador - Cap1.pdf")
with open("../Text/Dostoyevski, Fëdor - El jugador - Cap1.txt", "w", encoding="utf-8") as f:
    for page in doc:
        f.write(page.get_text() + "\n")
doc.close()

## Test 2 (From PDF to Audio)

In [ ]:
# Lee el archivo .txt y guárdalo en una variable como string
filename_plain_text = "Dostoyevski, Fëdor - El jugador - Cap1.txt"
with open(f"../Text/{filename_plain_text}", "r", encoding="utf-8") as f:
    texto_completo = f.read()

# 1. Dividir el texto en fragmentos (1500 caracteres para evitar pasarse del límite de tokens)
fragments = textwrap.wrap(texto_completo, 1500)
print(f"Text has been divided in {len(fragments)} fragments.")

# 2. Generar audios por fragmento
os.mkdir(f"../Audios/{filename_plain_text.split('.')[0]}")
files = []
for i, fragment in enumerate(fragments, start=1):
    response = client.audio.speech.create(
        model="gpt-4o-mini-tts",  # Modelo TTS
        voice="alloy",            # Cambia la voz si quieres
        input=fragment
    )

    filename = f"../Audios/{filename_plain_text.split('.')[0]}/part_{i}.mp3"
    with open(filename, "wb") as f:
        f.write(response.read())
    
    files.append(filename)
    print(f"{filename} saved")

# 3. Unir los audios en un solo archivo
combined = AudioSegment.empty()
for file in files:
    combined += AudioSegment.from_mp3(file)
    
combined.export(f"../Audios/{filename_plain_text.split('.')[0]}/audio_completo.mp3", format="mp3")
print("✅ Archivo final creado: audio_completo.mp3")


../Audios/Dostoyevski, Fëdor - El jugador - Cap1/part_1.mp3 saved
../Audios/Dostoyevski, Fëdor - El jugador - Cap1/part_2.mp3 saved
../Audios/Dostoyevski, Fëdor - El jugador - Cap1/part_3.mp3 saved
../Audios/Dostoyevski, Fëdor - El jugador - Cap1/part_4.mp3 saved
../Audios/Dostoyevski, Fëdor - El jugador - Cap1/part_5.mp3 saved
../Audios/Dostoyevski, Fëdor - El jugador - Cap1/part_6.mp3 saved
../Audios/Dostoyevski, Fëdor - El jugador - Cap1/part_7.mp3 saved
../Audios/Dostoyevski, Fëdor - El jugador - Cap1/part_8.mp3 saved
../Audios/Dostoyevski, Fëdor - El jugador - Cap1/part_9.mp3 saved
../Audios/Dostoyevski, Fëdor - El jugador - Cap1/part_10.mp3 saved
../Audios/Dostoyevski, Fëdor - El jugador - Cap1/part_11.mp3 saved
../Audios/Dostoyevski, Fëdor - El jugador - Cap1/part_12.mp3 saved
../Audios/Dostoyevski, Fëdor - El jugador - Cap1/part_13.mp3 saved
✅ Archivo final creado: texto_completo.mp3
